# Compare GPU Roofline Results

This notebook reloads and re-parses Nsight Compute report files for multiple GPUs, normalizes the data, and compares per-kernel roofline metrics (traffic, FLOPs, arithmetic intensity, and timing) across devices for both CUDA and OpenMP builds.

In [1]:
import pandas as pd
import os
import glob
import numpy as np
import seaborn as sns
import json
import matplotlib.pyplot as plt
import re
import sys
from tqdm import tqdm

sys.path.append('../')
from utils import *
from gatherData import _parse_ncu_report, roofline_results_to_df, calc_roofline_data 

from tqdm.contrib.concurrent import process_map
from functools import partial
import os
from os import path


from sass_helper import *

/Users/gbolet/miniconda3/envs/gpuflopbench-updated/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# kernel key and metrics
key_cols = ['source', 'Kernel Name', 'exeArgs', 'Block Size', 'Grid Size', 'model_type']
device_col = 'device'

groupings = key_cols + [device_col]
metrics = ['SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'traffic',
           'bytesRead', 'bytesWrite', 'bytesTotal',
           'dpAI', 'spAI', 'hpAI',
           'dpPerf', 'spPerf', 'hpPerf', 'xtime']

log_metrics = ['traffic', 'xtime']

# all cols
all_cols = groupings + metrics + ['sample']

# markers we should ignore / drop kernels containing these from the dataset
#library_markers = [ 'cub::', 'thrust::', '__cuda_' ]

In [3]:
df = pd.read_csv('all-NCU-GPU-Data.csv', low_memory=False)
print(df.shape)
print(df.columns)


# let's rename the device column
def rename_devices(x):
    if '3080' in x:
        return '3080'
    elif 'A100' in x:
        return 'A100'
    elif 'A10' in x:
        return 'A10'
    elif 'H100' in x:
        return 'H100'
    else:
        raise ValueError(f'Unknown device name in {x}')

df['device'] = df['device'].apply(lambda x: rename_devices(x))


(20044, 2659)
Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'smsp__sass_inst_executed_op_shared.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_shared_gmma.sum',
       'smsp__sass_inst_executed_op_shared_gmma.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_tma_ld.sum',
       'smsp__sass_inst_executed_op_tma_red.sum',
       'smsp__sass_inst_executed_op_tma_st.sum', 'launch_cluster_dim_x',
       'launch_cluster_dim_z', 'launch_cluster_max_active',
       'launch_cluster_dim_y'],
      dtype='object', length=2659)


In [4]:
sass = pd.read_csv('all-NCU-SASS-Data.csv', low_memory=False)
print(sass.shape)
print(sass.columns)
print(sass.columns[68], sass.columns[74])

(16606832, 76)
Index(['Address', 'Source', 'Warp Stall Sampling (All Samples)',
       'Warp Stall Sampling (Not-issued Samples)', '# Samples',
       'Instructions Executed', 'Thread Instructions Executed',
       'Predicated-On Thread Instructions Executed', 'Avg. Threads Executed',
       'Avg. Predicated-On Threads Executed',
       'Derivative Avg. Predicated-On Threads Executed',
       'Avg. Predicated-On Threads Unexecuted', 'Divergent Branches',
       'Address Space', 'Access Operation', 'Access Size',
       'L1 Tag Requests Global', 'L1 Conflicts Shared N-Way',
       'L1 Wavefronts Shared Excessive', 'L1 Wavefronts Shared',
       'L1 Wavefronts Shared Ideal', 'L2 Theoretical Sectors Global Excessive',
       'L2 Theoretical Sectors Global', 'L2 Theoretical Sectors Global Ideal',
       'L2 Explicit Evict Policies', 'L2 Explicit Hit Policy Evict First',
       'L2 Explicit Hit Policy Evict Last',
       'L2 Explicit Hit Policy Evict Normal',
       'L2 Explicit Hit Policy 

In [5]:
display(sass.head())

,Address,Source,Warp Stall Sampling (All Samples),Warp Stall Sampling (Not-issued Samples),# Samples,Instructions Executed,Thread Instructions Executed,Predicated-On Thread Instructions Executed,Avg. Threads Executed,Avg. Predicated-On Threads Executed,...,stall_wait (Not Issued),Kernel Name,device,codename,source,sample,model_type,Demangled Name,exeArgs,L2 Theoretical Sectors Local
0,0x7563ef27bb00,"MOV R1, c[0x0][0x28]",3,2,3.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
1,0x7563ef27bb10,"S2R R6, SR_CTAID.X",0,0,0.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
2,0x7563ef27bb20,"ULDC.64 UR4, c[0x0][0x118]",48,48,48.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
3,0x7563ef27bb30,"ISETP.GE.AND P0, PT, R6, c[0x0][0x160], PT",2,1,2.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
4,0x7563ef27bb40,"@P0 MOV R0, RZ",2,0,2.0,16384.0,524288.0,0.0,32.0,0.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN


In [6]:
sass_columns_to_keep = [
    'Kernel Name',
    'Address',
    'device',
    'codename',
    'exeArgs',
    'source',
    'Source',
    'sample',
    'model_type',
    'Demangled Name',
]
sass = sass[sass_columns_to_keep]

In [7]:
sass = sass.sort_values(by = ['codename', 'Kernel Name', 'device', 'exeArgs', 'source', 'sample', 'model_type', 'Demangled Name'], ignore_index=True)

# for each column, we need to make sure it's a string and strip it
for col in sass_columns_to_keep:
    sass[col] = sass[col].astype(str).str.strip()

In [8]:
# this elusive geglu kernel, let's see how many particualar samples of particular addresses we have
# for some strange reason, there is a whole sample that is repeated
subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ad400'))]

# for this subset, display which columns' are different



display(subset)

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name
4166904,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167376,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167848,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168320,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168792,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4169264,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."


In [9]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ac200'))]

display(subset)

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name
4166616,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167088,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167560,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168032,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168504,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168976,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."


In [10]:

display(sass.head(15))

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name
0,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb00,3080,accuracy,8192 10000 10 100,accuracy-cuda,"MOV R1, c[0x0][0x28]",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
1,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb10,3080,accuracy,8192 10000 10 100,accuracy-cuda,"S2R R6, SR_CTAID.X",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
2,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb20,3080,accuracy,8192 10000 10 100,accuracy-cuda,"ULDC.64 UR4, c[0x0][0x118]",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
3,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb30,3080,accuracy,8192 10000 10 100,accuracy-cuda,"ISETP.GE.AND P0, PT, R6, c[0x0][0x160], PT",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
4,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb40,3080,accuracy,8192 10000 10 100,accuracy-cuda,"@P0 MOV R0, RZ",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
5,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb50,3080,accuracy,8192 10000 10 100,accuracy-cuda,@P0 BRA 0x7563ef27bf00,1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
6,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb60,3080,accuracy,8192 10000 10 100,accuracy-cuda,"S2R R11, SR_TID.X",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
7,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb70,3080,accuracy,8192 10000 10 100,accuracy-cuda,"S2R R13, SR_LANEID",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
8,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb80,3080,accuracy,8192 10000 10 100,accuracy-cuda,"SHF.R.S32.HI R0, RZ, 0x1f, R11",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
9,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb90,3080,accuracy,8192 10000 10 100,accuracy-cuda,"LEA.HI R2, R0, R11, RZ, 0x5",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."


In [11]:
# for the sass dataframe, group it by 'Kernel Name', 'device', 'codename', 'exeArgs', and 'source'

# for some strange reason, there are duplicate rows, so we need to drop them first
sass['opcode'] = sass['Source'].apply(lambda x: extract_opcode_from_line(x))
sass['is_predicated'] = sass['Source'].apply(lambda x: detect_guard_pred_instruction(x))
sass['hex_refs'] = sass['Source'].apply(lambda x: extract_hex_references(x))

print(sass.shape)
sass = sass.drop_duplicates(ignore_index=True).reset_index(drop=True)
print(sass.shape)

sass_grouped = sass.groupby(['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'sample', 'model_type', 'Demangled Name'], dropna=False)


(16606832, 13)
(10574472, 13)


In [12]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ad400'))]

display(subset)

for col in subset.columns:
    print(subset[col].value_counts())

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name,opcode,is_predicated,hex_refs
2783680,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",1,cuda,"void geglu_kernel<float, float, (int)160, (int...",LOP3,True,0x80000000
2784152,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",2,cuda,"void geglu_kernel<float, float, (int)160, (int...",LOP3,True,0x80000000
2784624,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",3,cuda,"void geglu_kernel<float, float, (int)160, (int...",LOP3,True,0x80000000


Kernel Name
_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_    3
Name: count, dtype: int64
Address
0x7e301f7ad400    1
0x7bf83f7ad400    1
0x7e13db7ad400    1
Name: count, dtype: int64
device
H100    3
Name: count, dtype: int64
codename
geglu    3
Name: count, dtype: int64
exeArgs
100    3
Name: count, dtype: int64
source
geglu-cuda    3
Name: count, dtype: int64
Source
@P0   LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT    3
Name: count, dtype: int64
sample
1    1
2    1
3    1
Name: count, dtype: int64
model_type
cuda    3
Name: count, dtype: int64
Demangled Name
void geglu_kernel<float, float, (int)160, (int)1280, (int)8, (int)2>(T1 *, const T1 *)    3
Name: count, dtype: int64
opcode
LOP3    3
Name: count, dtype: int64
is_predicated
True    3
Name: count, dtype: int64
hex_refs
0x80000000    3
Name: count, dtype: int64


In [15]:
def get_hex_refs_of_group(group):
    raw_hex_refs = group['hex_refs'].unique()
    # split the hex refs if it contains multiple refs separated by ':'
    hex_refs = set()
    for hr in raw_hex_refs:
        hex_refs.update(hr.split(':'))

    return hex_refs

In [24]:

new_sasses = []

counter = 0
# for each group, let's analyze the SASS source code and create an instruction mix profile
# each group should be one particular kernel, we have 3 samples, so the output should have 3 rows per kernel
for name, group in tqdm(sass_grouped):
    #group = group.sort_values(by='Address') 

    this_group_hex_refs = get_hex_refs_of_group(group)
    this_kernel_name = name[0]
    this_device = name[1]
    this_codename = name[2]
    this_exeArgs = name[3]
    this_source = name[4]
    this_sample = name[5]
    this_model_type = name[6]

    if (this_codename != 'tensorT') and (this_model_type != 'cuda'):
        continue

    # for all the groups with the same 'device', 'codename', 'exeArgs', 'source', 'sample', 'model_type' but different 'Kernel Name'
    # need a faster way to do this than a for-loop
    filtered_groups = sass_grouped.filter(lambda x:
        (x.name[1] == this_device) and
        (x.name[2] == this_codename) and
        (x.name[3] == this_exeArgs) and
        (x.name[4] == this_source) and
        (x.name[5] == this_sample) and
        (x.name[6] == this_model_type) and
        (x.name[0] != this_kernel_name)
    )
    for other_name, other_group in filtered_groups.groupby(['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'sample', 'model_type', 'Demangled Name'], dropna=False):
        other_group_addresses = set(other_group['Address'].unique()) 

        # if there is an intersection between the hex refs and addresses of the other group, then we have found a reference
        if this_group_hex_refs.intersection(other_group_addresses):
            print(f"Found groups that references another group: {name} and {other_name}")
            print("------")
            break

    # create a histogram of the opcodes
    opcode_counts = group['opcode'].value_counts().to_dict()

    # prefix each key with 'opcode_'
    opcode_counts = {f'opcode_{k}': v for k, v in opcode_counts.items()}

    lines_of_code = group.shape[0]

    new_row = {
        'Kernel Name': this_kernel_name,
        'device': this_device,
        'codename': this_codename,
        'exeArgs': this_exeArgs,
        'source': this_source,
        'sample': this_sample,
        'model_type': this_model_type,
        'Demangled Name': name[7],
        'lines_of_code': lines_of_code,
        **opcode_counts
    }
    new_sasses.append(new_row)

    #print(new_row)


new_sass = pd.DataFrame(new_sasses)

# print(sass_grouped.head())

  1%|▏         | 310/22184 [04:24<5:11:30,  1.17it/s]


KeyboardInterrupt: 

In [ ]:
print(new_sass.head())

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

#subset = new_sass[(new_sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_')]

#display(subset)


In [ ]:
# sanity check, these opcodes should be identical across samples of the same kernel
new_sass_grouped = new_sass.groupby(['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'model_type', 'Demangled Name'], dropna=False)

condensed_imix = []

for name, group in tqdm(new_sass_grouped):
    if group.shape[0] > 1:
        group = group.sort_values(by='sample')

        mismatch_found = False

        # if the lines_of_code are different, then we have a problem
        first_loc = group.iloc[0]['lines_of_code']
        for idx, row in group.iterrows():
            if row['lines_of_code'] != first_loc:
                print("Lines of code mismatch found in group:", name)
                #display(group)
                mismatch_found = True
                break

        if mismatch_found:
            continue

    condensed_imix.append(group.iloc[0])

        # this still works, but is too verbose
        # check if all rows are identical
        #first_row = group.iloc[0]
        #first_row = first_row.drop('sample')
        #for idx, row in group.iterrows():
        #    # drop the sample index column for comparison
        #    if row.sample:
        #        row = row.drop('sample')
        #    if not row.equals(first_row):
        #        print("Mismatch found in group:", name)
        #        display(group)
        #        # print the columns that are different
        #        diff_cols = first_row[first_row != row].index.tolist()
        #        print("Different columns:", diff_cols)
        #        # print the values of the different columns
        #        for col in diff_cols:
        #            print(f"Column: {col}, idx: {idx}, Value 1: {first_row[col]}, Value 2: {row[col]}")
            #assert row.equals(first_row), "Mismatch found in group"


condensed_imix_df = pd.DataFrame(condensed_imix)

In [ ]:
# let's fix the omp kernel names 

def fix_omp_kernel_name(x):
    if '__omp_offloading' in x:
        # we need to do a regex here to extract the function name after the following 
        # possible string patterns:
        # __omp_offloading_fd01_2c7d38_
        # __omp_offloading_10305_2b800c7_
        match = re.search(r'__omp_offloading_.*?_.*?_(.+)$', x)
        if match:
            return match.group(1)
    return x
# the rest of the chars are the function name and line number -- which is the same across compilations
condensed_imix_df['Kernel Name'] = condensed_imix_df['Kernel Name'].apply(fix_omp_kernel_name)

In [ ]:
df['Kernel Name'] = df['Kernel Name'].apply(fix_omp_kernel_name)

In [ ]:
print(condensed_imix_df.shape)

print(condensed_imix_df['device'].value_counts())

In [ ]:
condensed_imix_df.to_csv('all-IMIX-Data.csv', index=False)


In [ ]:
print(df.shape)

# let's get overlapping column names
overlap_cols = list(set(df.columns).intersection(set(condensed_imix_df.columns)))
print("Overlapping columns:", overlap_cols)

# lets get lowered overlapping names
overlap_cols = list(set(df.columns.str.lower()).intersection(set(condensed_imix_df.columns.str.lower())))
print("Lowered Overlapping columns:", overlap_cols)

In [ ]:
# check device value counts
print(condensed_imix_df['device'].value_counts())
print(df['device'].value_counts())


In [ ]:
merge_cols = ['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'model_type', 'Demangled Name']
# now let's merge the imix data with the perf counter data
merged_df = pd.merge(df, condensed_imix_df, on=merge_cols, suffixes=('', '_imix'))

In [ ]:
print(merged_df.shape)

print(merged_df[merge_cols].head(10))

assert merged_df.columns.str.contains('opcode').any(), "No opcode columns found in merged dataframe"
assert merged_df.columns.str.contains('smsp').any(), "No perf counter columns found in merged dataframe"

In [ ]:
merged_df.to_csv('all-NCU-GPU-Data-with-IMIX.csv', index=False)